In [1]:
pip install chromadb scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import json

base_path = "RETAIL_EMBEDDING"

documents = []
metadatas = []
ids = []

X = []
y = []

for category_folder in os.listdir(base_path):

    folder_path = os.path.join(base_path, category_folder)

    if os.path.isdir(folder_path):

        for file in os.listdir(folder_path):

            if file.endswith(".json"):

                filepath = os.path.join(folder_path, file)

                with open(filepath, "r", encoding="utf-8") as f:
                    data = json.load(f)

                text = (
                    data.get("title", "") + " " +
                    data.get("description", "") + " " +
                    data.get("fullText", "") + " " +
                    data.get("retailUseCase", "") + " " +
                    data.get("targetAudience", "") + " " +
                    data.get("industryInsights", "")
                )

                X.append(text)
                y.append(data["category"])

                documents.append(text)

                metadatas.append({
                    "category": data["category"],
                    "fileId": data["fileId"]
                })

                ids.append(data["fileId"])

print("Documents:", len(documents))

Documents: 500


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

X_vectors = vectorizer.fit_transform(X)

classifier = LogisticRegression(
    max_iter=2000
)

classifier.fit(X_vectors, y)

print("Classifier trained")

Classifier trained


In [5]:
import pickle

pickle.dump(
    vectorizer,
    open("tfidf_vectorizer.pkl", "wb")
)

pickle.dump(
    classifier,
    open("category_classifier.pkl", "wb")
)

In [6]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    name="retail_documents"
)

In [18]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

document_embeddings = embedder.encode(
    documents,
    show_progress_bar=True
)

print(document_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

(500, 384)


In [19]:
collection.add(
    ids=ids,
    documents=documents,
    embeddings=document_embeddings.tolist(),
    metadatas=metadatas
)

In [8]:
query = "I want information about seasonal fashion trends"

In [20]:
query = "I want information about seasonal fashion trends"

query_vec = vectorizer.transform([query])

predicted_category = classifier.predict(query_vec)[0]

print(predicted_category)

Apparel


In [21]:
query_embedding = embedder.encode([query])[0]

In [22]:
import time

start = time.time()

results_no_filter = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

end = time.time()

print("Without Metadata:", end-start)

Without Metadata: 0.023027658462524414


In [23]:
start = time.time()

results_filter = collection.query(
    query_embeddings=[query_embedding.tolist()],
    where={
        "category": predicted_category
    },
    n_results=5
)

end = time.time()

print("With Metadata:", end-start)

With Metadata: 0.006128072738647461


In [24]:
for doc in results_no_filter["documents"][0]:
    print(doc[:200])
    print("-"*50)

Apparel Retail Insight 27 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
--------------------------------------------------
Apparel Retail Insight 29 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
--------------------------------------------------
Apparel Retail Insight 14 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
--------------------------------------------------
Apparel Retail Insight 64 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
-------------------------------------------

In [25]:
for doc in results_filter["documents"][0]:
    print(doc[:200])
    print("-"*50)

Apparel Retail Insight 27 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
--------------------------------------------------
Apparel Retail Insight 29 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
--------------------------------------------------
Apparel Retail Insight 14 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
--------------------------------------------------
Apparel Retail Insight 64 A comprehensive informational overview for apparel retail operations and customer engagement. Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and st
-------------------------------------------